# Session 2C — MCP Client (Auto-Discovered Tools)

**GOAL:** Show the 'USB-C for AI' concept.  
Instead of writing `@tool` decorators, we connect to an MCP Server and the agent **AUTO-DISCOVERS** all available tools at runtime.

Zero tool code in this file!

In [1]:
import os, sys, subprocess, warnings, logging, asyncio

warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

try:
    import utils.dns_patch
except Exception:
    pass

# ── Jupyter + Windows + Python 3.14 MCP Fix ─────────────────────────────
# Problem: MCP's stdio_client spawns a subprocess. On Windows + Python 3.14:
#   1) asyncio.create_subprocess_exec raises NotImplementedError (SelectorEventLoop)
#   2) The subprocess.Popen fallback passes Jupyter's sys.stderr as stderr=,
#      but Jupyter's OutStream doesn't support fileno() → UnsupportedOperation
#
# Fix: Monkey-patch stdio_client to always use subprocess.PIPE for stderr.
# This is applied to BOTH the mcp module and the langchain_mcp_adapters module
# (which imports stdio_client into its own namespace at import time).
import mcp.client.stdio as _mcp_stdio
import langchain_mcp_adapters.sessions as _lc_sessions

_original_stdio_client = _mcp_stdio.stdio_client

from contextlib import asynccontextmanager
@asynccontextmanager
async def _patched_stdio_client(server, errlog=None):
    """Wrapper that forces errlog=subprocess.PIPE to avoid Jupyter fileno() crash."""
    async with _original_stdio_client(server, errlog=subprocess.PIPE) as streams:
        yield streams

# Patch both modules so the fix takes effect everywhere
_mcp_stdio.stdio_client = _patched_stdio_client
_lc_sessions.stdio_client = _patched_stdio_client

from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

MCP_SERVER_SCRIPT = os.path.join(os.path.abspath('.'), 'mcp_server.py')
print('Setup complete.')
print(f'MCP Server: {MCP_SERVER_SCRIPT}')

Setup complete.
MCP Server: B:\Agentic_AI_Demos\session_2_frameworks\mcp_server.py


### Run the MCP Agent

We connect to the MCP server, auto-discover tools, and let the agent use them.

> **Note:** In Jupyter we can use `await` directly at the top level.

In [2]:
from utils.llm_router import get_routed_llm

async def run_mcp_agent():
    print('Connecting to MCP Server ...')
    client = MultiServerMCPClient({
        'inbox_server': {
            'command': sys.executable,
            'args': [MCP_SERVER_SCRIPT],
            'transport': 'stdio',
        }
    })

    tools = await client.get_tools()
    print(f'\nAuto-discovered {len(tools)} tools:')
    for t in tools:
        print(f'   {t.name}: {t.description[:80]}')

    llm = get_routed_llm(role='worker_model')
    agent = create_react_agent(llm, tools)

    print('\nRunning agent with MCP-discovered tools ...\n')
    result = await agent.ainvoke({
        'messages': [
            HumanMessage(content=(
                'First get my inbox statistics, then fetch my recent '
                'emails and analyze them. If you find any meeting '
                'requests, schedule them on my calendar. '
                'Give me a complete summary at the end.'
            ))
        ]
    })

    print('FULL AGENT CONVERSATION:')
    print('-' * 50)
    for msg in result['messages']:
        role = msg.type.upper()
        content = getattr(msg, 'content', '')
        if isinstance(content, list):
            content = '\n'.join(c.get('text', '') if isinstance(c, dict) and 'text' in c else str(c) for c in content)
        if content:
            print(f'\n[{role}]:')
            print(f'  {content}')
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f'  Tool Call: {tc["name"]}({tc["args"]})')

    print('\nKEY TAKEAWAY:')
    print('   We wrote ZERO tool definitions in this file!')
    print('   MCP = USB-C for AI')

await run_mcp_agent()

Connecting to MCP Server ...

Auto-discovered 3 tools:
   fetch_inbox: Fetch the most recent emails from the user's Gmail inbox.

Args:
    limit: Numb
   schedule_calendar_event: Schedule a meeting on Google Calendar and send invite emails with a Google Meet 
   get_inbox_stats: Get quick statistics about the user's inbox.

Returns:
    A summary string with

Running agent with MCP-discovered tools ...

FULL AGENT CONVERSATION:
--------------------------------------------------

[HUMAN]:
  First get my inbox statistics, then fetch my recent emails and analyze them. If you find any meeting requests, schedule them on my calendar. Give me a complete summary at the end.
  Tool Call: get_inbox_stats({})
  Tool Call: fetch_inbox({})

[TOOL]:
  Inbox Stats: 10 recent emails, 1 urgent, 1 meeting-related, 8 informational.

[TOOL]:
  
Email 1:
  From:    Akshat Dodwad <akshatdodwad@gmail.com>
  Subject: Meeting on 25th April 2026 at 11 am
  Preview: We will conduct one meeting on 25th April 202